# Credit Card Approval Prediction - Exploratory Data Analysis (EDA)
This notebook contains the Exploratory Data Analysis for the Credit Card Approval Prediction system. 
We investigate applicant demographics and credit status behavior to understand patterns and check feature distributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Ensure project root is in the system path to access configuration
sys.path.append(str(Path.cwd().parent))
from config import get_config

config = get_config()
sns.set_theme(style="whitegrid", context="talk")
print("Setup Complete!")

## 1. Load Datasets
We load both `application_record.csv` (applicant profiles) and `credit_record.csv` (payment history records).

In [ ]:
app_df = pd.read_csv(config.APPLICATION_RECORD_PATH)
credit_df = pd.read_csv(config.CREDIT_RECORD_PATH)

print(f"Application Records Shape: {app_df.shape}")
print(f"Credit Records Shape: {credit_df.shape}")

## 2. Process Binary Target Class
The bank status categories in `credit_record.csv` represent payment delay windows. We map these to binary classifications:
- **Good Payment (STATUS = 0, X, C)**: Approved (1)
- **Bad Payment (STATUS = 1, 2, 3, 4, 5)**: Rejected (0)

If an applicant has any month classified as bad status, their card application is rejected.

In [ ]:
bad_status = {'1', '2', '3', '4', '5'}
credit_df['is_bad'] = credit_df['STATUS'].apply(lambda x: 1 if x in bad_status else 0)

# Aggregate payment records by ID: if any month has bad payment -> Rejected (0), else Approved (1)
grouped = credit_df.groupby('ID')['is_bad'].sum().reset_index()
grouped['STATUS_LABEL'] = grouped['is_bad'].apply(lambda x: 0 if x > 0 else 1)
labels_df = grouped[['ID', 'STATUS_LABEL']]

# Merge demographics with target label
merged_df = pd.merge(app_df, labels_df, on='ID', how='inner')
print(f"Merged Dataset Shape: {merged_df.shape}")

## 3. Explanatory Visualizations

### 3.1 Credit Approval Status (Target Variable) Distribution

In [ ]:
plt.figure(figsize=(6, 6))
counts = merged_df['STATUS_LABEL'].value_counts()
labels = ['Approved (Good)', 'Rejected (Bad)']
colors = ['#2ca02c', '#d62728']

plt.pie(counts, labels=labels, autopct='%1.2f%%', startangle=90, colors=colors, explode=(0, 0.1))
plt.title("Target Class Distribution (STATUS_LABEL)")
plt.show()

### 3.2 Income Distribution
Let's look at the distribution of annual income among the applicants, trimming outliers (above the 99th percentile) for visual clarity.

In [ ]:
plt.figure(figsize=(10, 5))
upper_limit = merged_df['AMT_INCOME_TOTAL'].quantile(0.99)
filtered_income = merged_df[merged_df['AMT_INCOME_TOTAL'] <= upper_limit]['AMT_INCOME_TOTAL']

sns.histplot(filtered_income, kde=True, color="teal", bins=30)
plt.title("Distribution of Annual Income (Filtered Outliers < 99th Pct)")
plt.xlabel("Annual Income (USD)")
plt.ylabel("Count")
plt.show()

### 3.3 Occupation and Housing Type Distribution

In [ ]:
plt.figure(figsize=(12, 6))
occ_df = merged_df.copy()
occ_df['OCCUPATION_TYPE'] = occ_df['OCCUPATION_TYPE'].fillna("Unspecified")
order = occ_df['OCCUPATION_TYPE'].value_counts().index
sns.countplot(data=occ_df, y='OCCUPATION_TYPE', order=order, palette="viridis")
plt.title("Distribution of Occupations")
plt.xlabel("Count")
plt.ylabel("Occupation")
plt.show()

plt.figure(figsize=(10, 5))
order_house = merged_df['NAME_HOUSING_TYPE'].value_counts().index
sns.countplot(data=merged_df, y='NAME_HOUSING_TYPE', order=order_house, palette="Set2")
plt.title("Housing Types of Applicants")
plt.xlabel("Count")
plt.ylabel("Housing Type")
plt.show()

### 3.4 Education Level & Family Status Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
edu_order = merged_df['NAME_EDUCATION_TYPE'].value_counts().index
sns.countplot(data=merged_df, y='NAME_EDUCATION_TYPE', order=edu_order, ax=axes[0], palette="Blues_r")
axes[0].set_title("Education Levels")
axes[0].set_xlabel("Count")

fam_order = merged_df['NAME_FAMILY_STATUS'].value_counts().index
sns.countplot(data=merged_df, x='NAME_FAMILY_STATUS', order=fam_order, ax=axes[1], palette="Set3")
axes[1].set_title("Family Status")
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_xlabel("Status")
plt.show()

### 3.5 Demographic Boxplots vs Credit Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
merged_df['AGE'] = (-merged_df['DAYS_BIRTH']) / 365.25
merged_df['STATUS_TEXT'] = merged_df['STATUS_LABEL'].map({0: 'Rejected', 1: 'Approved'})

sns.boxplot(data=merged_df[merged_df['AMT_INCOME_TOTAL'] <= upper_limit], x='STATUS_TEXT', y='AMT_INCOME_TOTAL', ax=axes[0], palette={'Rejected': '#d62728', 'Approved': '#2ca02c'})
axes[0].set_title("Annual Income vs Status")
axes[0].set_ylabel("Income (USD)")

sns.boxplot(data=merged_df, x='STATUS_TEXT', y='AGE', ax=axes[1], palette={'Rejected': '#d62728', 'Approved': '#2ca02c'})
axes[1].set_title("Age vs Status")
axes[1].set_ylabel("Age (Years)")
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
num_cols = merged_df.select_dtypes(include=[np.number]).columns.tolist()
if 'ID' in num_cols:
    num_cols.remove('ID')

sns.heatmap(merged_df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
plt.title("Correlation Heatmap of Numerical Features")
plt.show()